# 05 · Cây quyết định

Chạy lại lựa chọn tham số cho hai luồng trên tập train. Test không được dùng trong notebook này.

Notebook chạy độc lập từ thư mục gốc hoặc `notebooks/`; cài `requirements-demo.txt` trước.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT/'student_success').exists(): ROOT = ROOT.parent
assert (ROOT/'student_success').exists(), 'Chạy từ thư mục dự án hoặc notebooks/'
sys.path.insert(0, str(ROOT))
from student_success.data import CATEGORICAL, NUMERIC, LABELS, SEED, split
DATA = ROOT/'data/raw/data.csv'
assert DATA.exists(), 'Chạy python scripts/download_data.py'
df = pd.read_csv(DATA, sep=';'); df.columns = df.columns.str.strip()
train_idx, test_idx = split(df)
train = df.loc[train_idx]
R = ROOT/'data/experiment'


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from student_success.preprocess import build_preprocessor
X = train[CATEGORICAL+NUMERIC]; y = train.Target
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
searches = {}
for flow in ['A','B']:
    pipeline = Pipeline([('preprocess', build_preprocessor(flow)), ('model', DecisionTreeClassifier(random_state=SEED))])
    search = GridSearchCV(pipeline, {"model__max_depth": [3,8,12], "model__min_samples_leaf": [1,10], "model__class_weight": [None,"balanced"]}, cv=folds, scoring='f1_macro', n_jobs=1)
    search.fit(X,y); searches[flow] = search
    print(flow, 'tham số:', search.best_params_, '| Macro-F1 CV:', round(search.best_score_,4))

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
for flow,search in searches.items():
    cv=pd.DataFrame(search.cv_results_)
    param='param_model__n_neighbors' if 'param_model__n_neighbors' in cv else 'param_model__max_depth'
    grouped=cv.groupby(param).mean_test_score.max()
    ax.plot(grouped.index,grouped.values,marker='o',label='Luồng '+flow)
ax.set(title='Macro-F1 CV tốt nhất theo k hoặc độ sâu',ylabel='Macro-F1 · 5 fold',xlabel='Giá trị tham số',ylim=(.34,.50)); ax.grid(axis='y',alpha=.25); ax.legend(); plt.show()
